In [1]:
REWARD_ROUNDING_THRESHOLD = 0.6

# Transitions

In [2]:
import time
import json
import os

file_path = "results/visualize_last_run.txt"

if not os.path.exists(file_path):
    with open(file_path, "w") as f:
        json.dump({"last_run": time.time() - 300}, f)

with open(file_path, "r") as f:
    data = json.load(f)

VISUALIZE_MODIF_TIME = data.get("last_run", time.time() - 300)

data["last_run"] = time.time()
with open(file_path, "w") as f:
    json.dump(data, f)

In [3]:
import json
def analyze_reward_transitions(json_path, first_k):
    """
    Analyzes reward transitions in a training log.

    Args:
        json_path (str): Path to the training_log.json file.
        first_k (int): Number of initial epochs to consider as 'early'.

    Returns:
        count_lost_interest (int): Number of docs that got reward=1 early but never again later.
        count_gained_interest (int): Number of docs that got no reward=1 early but did later.
        lost_interest_titles (list): Titles of papers for the first category.
        gained_interest_titles (list): Titles of papers for the second category.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Handle both list and dict formats
    if isinstance(data, list):
        epochs_data = {}
        for record in data:
            epoch = record.get("epoch", 0)
            if epoch not in epochs_data:
                epochs_data[epoch] = []
            paper = {
                "title of paper": f"paper_{record.get('record_id', 'unknown')}",
                "generated_ideas": [{
                    "generated_idea": record.get("completion", ""),
                    "reward": record.get("reward", 0)
                }]
            }
            epochs_data[epoch].append(paper)
        epoch_data = epochs_data
    else:
        epoch_data = data["epochs"]  # Dict[str, List[Dict]] where str is epoch number
    
    # Aggregate rewards per title
    reward_map = {}  # title -> {epoch -> [rewards]}
    
    for epoch_str, paper_entries in epoch_data.items():
        epoch = int(epoch_str)
        for paper in paper_entries:
            title = paper["title of paper"]
            rewards = [idea["reward"] for idea in paper["generated_ideas"]]
            reward_map.setdefault(title, {}).setdefault(epoch, []).extend(rewards)

    lost_interest_titles = []
    gained_interest_titles = []

    for title, epoch_rewards in reward_map.items():
        early_rewards = []
        late_rewards = []

        for epoch, rewards in epoch_rewards.items():
            if epoch <= first_k:
                early_rewards.extend(rewards)
            else:
                late_rewards.extend(rewards)

        early_has_1 = any(r == 1 for r in early_rewards)
        late_has_1 = any(r == 1 for r in late_rewards)

        if early_has_1 and not late_has_1:
            lost_interest_titles.append(title)
        elif not early_has_1 and late_has_1:
            gained_interest_titles.append(title)

    return (
        len(lost_interest_titles),
        len(gained_interest_titles),
        lost_interest_titles,
        gained_interest_titles
    )

In [4]:
import os
import matplotlib.pyplot as plt

def plot_reward_transitions_combined(json_path, plots_dir):
    """
    Plots a single line chart showing how reward transitions (gained/lost) vary across different epochs.

    Args:
        json_path (str): Path to train_logs.json.
        plots_dir (str): Output directory to save the plot.
    """
    gained_counts = []
    lost_counts = []
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Handle both list and dict formats
    if isinstance(data, list):
        epochs_data = {}
        for record in data:
            epoch = record.get("epoch", 0)
            if epoch not in epochs_data:
                epochs_data[epoch] = []
            paper = {
                "title of paper": f"paper_{record.get('record_id', 'unknown')}",
                "generated_ideas": [{
                    "generated_idea": record.get("completion", ""),
                    "reward": record.get("reward", 0)
                }]
            }
            epochs_data[epoch].append(paper)
        data = {"epochs": epochs_data}
    
    epochs = list(range(len(data["epochs"])))

    for epoch in epochs:
        lost_count, gained_count, _, _ = analyze_reward_transitions(json_path, epoch)
        lost_counts.append(lost_count)
        gained_counts.append(gained_count)

    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(12, 6))
    plt.plot(epochs, gained_counts, marker='o', color='green', label='Gained Reward')
    plt.plot(epochs, lost_counts, marker='s', color='red', label='Lost Reward')
    plt.ylim(0, max(max(gained_counts), max(lost_counts))*1.1)

    plt.xlabel("Epoch Split (first_k)")
    plt.ylabel("Number of Documents")
    plt.suptitle(json_path.split("/")[1])
    plt.title("📈 Reward Transitions vs Epoch Split")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()

    save_path = os.path.join(plots_dir, "reward_transitions_combined.png")
    plt.savefig(save_path)
    plt.close()
    print(f"✅ Saved combined reward transitions plot to {save_path}")


# Reward Distribution

In [5]:
import os
import re
import json
import matplotlib.pyplot as plt

def plot_reward_distribution_per_step(json_path, output_dir, max_steps=100):
    """
    Creates a box plot showing reward distributions per step (batch),
    where each generation (idea) is treated as a single reward point.

    Args:
        json_path (str): Path to train_logs.json.
        output_dir (str): Directory to save the plot.
        max_steps (int): Max number of steps to show (for readability).
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    epochs_data = data.get("epochs", {})
    os.makedirs(output_dir, exist_ok=True)

    # Extract batch size from path (e.g. "b8")
    match = re.search(r'_b(\d+)_', json_path)
    if not match:
        print("❌ Could not extract batch size from path.")
        return
    batch_size = int(match.group(1))

    # --- Collect all rewards individually (per generation)
    all_rewards = []
    for epoch in epochs_data.values():
        for paper in epoch:
            for idea in paper.get("generated_ideas", []):
                reward = idea.get("reward", None)
                if reward is not None:
                    all_rewards.append(reward)

    if not all_rewards:
        print("❌ No reward data found.")
        return

    # --- Group into batches (steps)
    reward_batches = [
        all_rewards[i:i + batch_size]
        for i in range(0, len(all_rewards), batch_size)
    ]

    # Truncate if too many steps
    if len(reward_batches) > max_steps:
        reward_batches = reward_batches[:max_steps]
        print(f"⚠️ Truncated to first {max_steps} steps for display.")

    # --- Create box plot
    plt.figure(figsize=(max(12, len(reward_batches) * 0.3), 6))

    boxplot = plt.boxplot(
        reward_batches,
        showfliers=True,
        patch_artist=True,  # Required for coloring boxes
        boxprops=dict(color='black'),
        medianprops=dict(color='black'),
        whiskerprops=dict(color='black'),
        capprops=dict(color='black'),
        flierprops=dict(marker='o', markerfacecolor='white', markeredgecolor='black', markersize=5)
    )

    # Set the face color for each box
    for patch in boxplot['boxes']:
        patch.set_facecolor('lightgray')

    plt.xlabel("Step (Batch Index)")
    plt.ylabel("Reward Distribution")
    plt.suptitle(json_path.split("/")[1])
    plt.title("📦 Reward Variance per Step (Box Plot)")
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    plot_path = os.path.join(output_dir, "reward_variance_boxplot.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"✅ Saved reward variance box plot to {plot_path}")


# Reward Variance

In [6]:
import os
import re
import json
import matplotlib.pyplot as plt

def plot_reward_variance_line(json_path, output_dir):
    """
    Plots reward variance per step (batch) as a line with shaded area.

    Each generation (idea) is treated as one reward sample.
    Steps correspond to batches based on batch size inferred from filename.

    Args:
        json_path (str): Path to training_log.json.
        output_dir (str): Directory to save the plot.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    epochs_data = data.get("epochs", {})
    os.makedirs(output_dir, exist_ok=True)

    # Extract batch size from path (e.g. "b8")
    match = re.search(r'_b(\d+)_', json_path)
    if not match:
        print("❌ Could not extract batch size from path.")
        return
    batch_size = int(match.group(1))

    # --- Collect all rewards individually (per generation)
    all_rewards = []
    for epoch in epochs_data.values():
        for paper in epoch:
            for idea in paper.get("generated_ideas", []):
                reward = idea.get("reward", None)
                if reward is not None:
                    all_rewards.append(reward)

    if not all_rewards:
        print("❌ No reward data found.")
        return

    # --- Group into batches (steps)
    reward_batches = [
        all_rewards[i:i + batch_size]
        for i in range(0, len(all_rewards), batch_size)
    ]

    # --- Compute variance per step
    import numpy as np
    variances = [np.var(batch) if len(batch) > 1 else 0.0 for batch in reward_batches]
    steps = list(range(len(variances)))

    # --- Plot with shaded area
    plt.figure(figsize=(14, 6))
    variances_np = np.array(variances)
    plt.plot(steps, variances_np, label="Reward Variance", color="darkblue", linewidth=2)
    plt.ylim(0, max(variances_np)*1.1)
    plt.fill_between(steps, variances_np, alpha=0.2, color="skyblue")

    plt.xlabel("Step (Batch Index)")
    plt.ylabel("Variance of Rewards")
    plt.suptitle(json_path.split("/")[1])
    plt.title("📈 Reward Variance per Step (Line + Shaded Area)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()

    plot_path = os.path.join(output_dir, "reward_variance_lineplot.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"✅ Saved reward variance line plot to {plot_path}")


# Training Stats

In [7]:
import os
import json
import re
import math
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

def plot_all_training_stats(json_path, output_dir):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    os.makedirs(output_dir, exist_ok=True)
    
    match = re.search(r'_b(\d+)_', json_path)
    if not match:
        print("❌ Could not extract batch size from path.")
        return
    batch_size = int(match.group(1))

    # Check if data is a list (new format) or dict (old format)
    if isinstance(data, list):
        # New format: data is a flat list of records
        # Group by (epoch, record_id) to reconstruct papers with multiple ideas
        from collections import defaultdict
        papers_by_epoch = defaultdict(lambda: defaultdict(list))
        
        for record in data:
            epoch = record.get("epoch", 0)
            record_id = record.get("record_id", "unknown")
            
            idea = {
                "generated_idea": record.get("completion", ""),
                "reward": record.get("reward", 0)
            }
            papers_by_epoch[epoch][record_id].append(idea)
        
        # Convert to the expected format
        epochs_data = {}
        for epoch, papers_dict in papers_by_epoch.items():
            epochs_data[epoch] = []
            for record_id, ideas in papers_dict.items():
                paper = {
                    "title of paper": f"paper_{record_id}",
                    "generated_ideas": ideas
                }
                epochs_data[epoch].append(paper)
        
        data = {"epochs": epochs_data, "losses": {}}

    # 1. Training Loss per Step
    plt.figure(figsize=(12, 5))
    df = pd.DataFrame()
    for epoch, loss_records in data.get("losses", {}).items():
        if loss_records:  # Only process if there are loss records
            df = pd.concat((df, pd.DataFrame(loss_records)), ignore_index=True)
        
    step_avg_losses = []
    for i in range(0, len(df), batch_size):
        batch = df['loss'][i:i + batch_size]
        step_avg = sum(batch) / len(batch)
        step_avg_losses.append(step_avg)
    
    if not df.empty:
        plt.plot(list(range(len(step_avg_losses))), step_avg_losses, color="black", alpha=0.4)
        plt.ylim(0, max(step_avg_losses)*1.1)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.suptitle(json_path.split("/")[1])
    plt.title("Training Loss per Step (All Epochs)")
    plt.grid(True)
    plt.tight_layout()
    loss_path = os.path.join(output_dir, "training_loss.png")
    plt.savefig(loss_path)
    print(f"✅ Saved training loss plot to {loss_path}")
    plt.close()
    
    # 2. Average Format Following per Epoch
    avg_formats = []
    epochs = []
    for epoch_str, papers in data.get("epochs", {}).items():
        formats = [1 if idea["generated_idea"] else 0 for paper in papers for idea in paper["generated_ideas"]]
        if formats:
            avg_formats.append(sum(formats) / len(formats))
            epochs.append(int(epoch_str))
    if epochs:
        plt.figure(figsize=(12, 6))
        plt.plot(epochs, avg_formats, marker='o')
        plt.ylim(0, max(avg_formats)*1.1)
        plt.xlabel("Epoch")
        plt.ylabel("Average Format Following")
        plt.suptitle(json_path.split("/")[1])
        plt.title("Average Format Following per Epoch")
        plt.grid(True)
        plt.tight_layout()
        format_path = os.path.join(output_dir, "avg_training_format.png")
        plt.savefig(format_path)
        print(f"✅ Saved training format plot to {format_path}")
        plt.close()

    # 3. Average Reward per Epoch
    avg_rewards = []
    epochs = []
    for epoch_str, papers in data.get("epochs", {}).items():
        rewards = [idea["reward"] for paper in papers for idea in paper["generated_ideas"]]
        if rewards:
            avg_rewards.append(sum(rewards) / len(rewards))
            epochs.append(int(epoch_str))
    if epochs:
        plt.figure(figsize=(12, 6))
        plt.plot(epochs, avg_rewards, marker='o')
        plt.ylim(0, max(avg_rewards)*1.1)
        plt.xlabel("Epoch")
        plt.ylabel("Average Reward")
        plt.suptitle(json_path.split("/")[1])
        plt.title("Average Reward per Epoch")
        plt.grid(True)
        plt.tight_layout()
        reward_path = os.path.join(output_dir, "avg_training_reward.png")
        plt.savefig(reward_path)
        print(f"✅ Saved training reward plot to {reward_path}")
        plt.close()

    # 3.1. Average Rounded Reward per Epoch
    avg_rewards = []
    epochs = []
    integer_reward = True
    for epoch_str, papers in data.get("epochs", {}).items():
        rewards = []
        for paper in papers:
            for idea in paper["generated_ideas"]:
                rewards.append(1 if idea["reward"]>REWARD_ROUNDING_THRESHOLD else 0)
                if idea["reward"] and idea["reward"]<1:
                    integer_reward = False
        if rewards:
            avg_rewards.append(sum(rewards) / len(rewards))
            epochs.append(int(epoch_str))
    if epochs and not integer_reward:
        plt.figure(figsize=(12, 6))
        plt.plot(epochs, avg_rewards, marker='o')
        plt.ylim(0, max(avg_rewards)*1.1)
        plt.xlabel("Epoch")
        plt.ylabel("Average Rounded Reward")
        plt.suptitle(json_path.split("/")[1])
        plt.title("Average Rounded Reward per Epoch")
        plt.grid(True)
        plt.tight_layout()
        reward_path = os.path.join(output_dir, "avg_training_rounded_reward.png")
        plt.savefig(reward_path)
        print(f"✅ Saved training rounded reward plot to {reward_path}")
        plt.close()

    # 4. Average BoN Reward per Epoch
    bon_rewards = []
    epochs = []
    for epoch_str, papers in data.get("epochs", {}).items():
        rewards = [max([idea["reward"] for idea in paper["generated_ideas"]]) for paper in papers]
        if rewards:
            bon_rewards.append(sum(rewards) / len(rewards))
            epochs.append(int(epoch_str))
    if epochs:
        plt.figure(figsize=(12, 6))
        plt.plot(epochs, bon_rewards, marker='o')
        plt.ylim(0, max(bon_rewards)*1.1)
        plt.xlabel("Epoch")
        plt.ylabel("Average BoN Reward")
        plt.suptitle(json_path.split("/")[1])
        plt.title("Average BoN Reward per Epoch")
        plt.grid(True)
        plt.tight_layout()
        reward_path = os.path.join(output_dir, "avg_training_bon_reward.png")
        plt.savefig(reward_path)
        print(f"✅ Saved training reward plot to {reward_path}")
        plt.close()
    
    # 4.1. Average BoN Rounded Reward per Epoch
    bon_rewards = []
    epochs = []
    integer_reward = True
    for epoch_str, papers in data.get("epochs", {}).items():
        rewards = []
        for paper in papers:
            paper_rewards = []
            for idea in paper["generated_ideas"]:
                paper_rewards.append(1 if idea["reward"]>REWARD_ROUNDING_THRESHOLD else 0)
                if idea["reward"] and idea["reward"]<1:
                    integer_reward = False
            rewards.append(max(paper_rewards))
        if rewards:
            bon_rewards.append(sum(rewards) / len(rewards))
            epochs.append(int(epoch_str))
    if epochs and not integer_reward:
        plt.figure(figsize=(12, 6))
        plt.plot(epochs, bon_rewards, marker='o')
        plt.ylim(0, max(bon_rewards)*1.1)
        plt.xlabel("Epoch")
        plt.ylabel("Average BoN Rounded Reward")
        plt.suptitle(json_path.split("/")[1])
        plt.title("Average BoN Rounded Reward per Epoch")
        plt.grid(True)
        plt.tight_layout()
        reward_path = os.path.join(output_dir, "avg_training_bon_rounded_reward.png")
        plt.savefig(reward_path)
        print(f"✅ Saved training rounded reward plot to {reward_path}")
        plt.close()

    # 5. Combined Reward Stats
    total_reward_counts = {}
    unique_doc_counts = {}
    for epoch_str, paper_entries in data["epochs"].items():
        epoch = int(epoch_str)
        total_count = 0
        unique_docs_with_reward = 0
        for paper in paper_entries:
            got_reward = False
            for idea in paper["generated_ideas"]:
                if idea["reward"] == 1:
                    total_count += 1
                    got_reward = True
            if got_reward:
                unique_docs_with_reward += 1
        total_reward_counts[epoch] = total_count
        unique_doc_counts[epoch] = unique_docs_with_reward

    sorted_epochs = sorted(total_reward_counts.keys())
    total_counts = [total_reward_counts[e] for e in sorted_epochs]
    unique_counts = [unique_doc_counts[e] for e in sorted_epochs]

    plt.figure(figsize=(12, 6))
    plt.plot(sorted_epochs, total_counts, marker='o', linestyle='-', color='blue', label='Total reward=1 ideas')
    plt.plot(sorted_epochs, unique_counts, marker='s', linestyle='--', color='green', label='Unique docs with reward=1')
    plt.ylim(0, max(max(total_counts), max(unique_counts))*1.1)
    plt.xlabel("Epoch")
    plt.ylabel("Count")
    plt.suptitle(json_path.split("/")[1])
    plt.title("Reward Statistics per Epoch")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    combined_path = os.path.join(output_dir, "combined_training_reward_stats.png")
    plt.savefig(combined_path)
    print(f"✅ Saved training combined reward stats plot to {combined_path}")
    plt.close()


def plot_all_validation_stats(json_path, output_dir):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    os.makedirs(output_dir, exist_ok=True)
    
    match = re.search(r'_b(\d+)_', json_path)
    if not match:
        print("⚠️ Could not extract batch size from path, using default value of 8.")
        batch_size = 8
    else:
        batch_size = int(match.group(1))

    # 1. Average Reward per Run
    avg_rewards = []
    runs = []
    for run_str, records in data.items():
        # Validation data structure: {epoch: [records]} where each record has 'reward' directly
        rewards = [record["reward"] for record in records if "reward" in record]
        if rewards:
            avg_rewards.append(sum(rewards) / len(rewards))
            runs.append(float(run_str))
    if runs:
        plt.figure(figsize=(12, 6))
        plt.plot(runs, avg_rewards, marker='o')
        plt.ylim(0, max(avg_rewards)*1.1)
        plt.xlabel("Epoch")
        plt.ylabel("Average Reward")
        plt.suptitle(json_path.split("/")[1])
        plt.title("Average Reward per Epoch")
        plt.grid(True)
        plt.tight_layout()
        reward_path = os.path.join(output_dir, "avg_validation_reward.png")
        plt.savefig(reward_path)
        print(f"✅ Saved validation reward plot to {reward_path}")
        plt.close()

def plot_avg_and_moving_reward_per_step(json_path, output_dir, window_size=10):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Handle both list and dict formats
    if isinstance(data, list):
        epochs_data = {}
        for record in data:
            epoch = record.get("epoch", 0)
            if epoch not in epochs_data:
                epochs_data[epoch] = []
            paper = {
                "title of paper": f"paper_{record.get('record_id', 'unknown')}",
                "generated_ideas": [{
                    "generated_idea": record.get("completion", ""),
                    "reward": record.get("reward", 0)
                }]
            }
            epochs_data[epoch].append(paper)
    else:
        epochs_data = data.get("epochs", {})
    
    os.makedirs(output_dir, exist_ok=True)

    match = re.search(r'_b(\d+)_', json_path)
    if not match:
        print("⚠️ Could not extract batch size from path, using default value of 8.")
        batch_size = 8
    else:
        batch_size = int(match.group(1))

    all_sample_rewards = []
    for epoch, papers in epochs_data.items():
        for paper in papers:
            ideas = paper.get("generated_ideas", [])
            if not ideas:
                continue
            rewards = [idea["reward"] for idea in ideas]
            if rewards:
                avg_reward = sum(rewards) / len(rewards)
                all_sample_rewards.append(avg_reward)

    if not all_sample_rewards:
        print("❌ No rewards found in train_logs.")
        return

    step_avg_rewards = []
    for i in range(0, len(all_sample_rewards), batch_size):
        batch = all_sample_rewards[i:i + batch_size]
        step_avg = sum(batch) / len(batch)
        step_avg_rewards.append(step_avg)

    df = pd.DataFrame({
        "step": list(range(len(step_avg_rewards))),
        "avg_reward": step_avg_rewards
    })
    df["moving_avg"] = df["avg_reward"].rolling(window=window_size).mean().dropna()

    plt.figure(figsize=(12, 6))
    plt.plot(df["step"], df["avg_reward"], label="Avg Reward per Step", color='blue', alpha=0.6)
    plt.plot(df["step"], df["moving_avg"], label=f"Moving Avg (window={window_size})", color='red')
    plt.ylim(0, max(max(df["avg_reward"]), max(df["moving_avg"]))*1.1)
    plt.xlabel("Step")
    plt.ylabel("Average Reward")
    plt.suptitle(json_path.split("/")[1])
    plt.title("Average Reward per Step with Moving Average")
    plt.legend()
    plt.grid(True)

    plot_path = os.path.join(output_dir, "reward_per_step.png")
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.close()
    print(f"✅ Saved reward-per-step plot to {plot_path}")

def plot_all_metrics(log_history, save_path):
    metrics = defaultdict(lambda: {'steps': [], 'values': []})

    exclude_keys = {'epoch', 'step', 'total_flos', 'train_runtime', 'train_samples_per_second', 'train_steps_per_second',
                    'train_loss', 'loss'}

    for log in log_history:
        if "step" not in log:
            continue
        step = log["step"]
        for key, value in log.items():
            if key not in exclude_keys and isinstance(value, (int, float)):
                metrics[key]['steps'].append(step)
                metrics[key]['values'].append(value)

    num_metrics = len(metrics)
    ncols = min(2, num_metrics)
    nrows = math.ceil(num_metrics / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows), sharex=True, squeeze=False)
    axes_flat = axes.flatten()

    for ax, (metric_name, data) in zip(axes_flat, metrics.items()):
        ax.plot(data['steps'], data['values'], marker='o' if 'eval' in metric_name else '.')
        ax.set_title(metric_name.replace("_", " "))
        ax.set_ylabel(metric_name.split('_')[-1])
        ax.grid(True)
        ax.set_xlabel("Steps")

    for i in range(num_metrics, len(axes_flat)):
        axes_flat[i].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path)
    print(f"📈 Plot saved to: {save_path}")
    plt.close()

def plot_from_latest_checkpoint(run_dir, output_dir):
    checkpoint_root = os.path.join(run_dir, "checkpoint")
    if not os.path.exists(checkpoint_root):
        print(f"❌ No checkpoint/ directory found in: {run_dir}")
        return

    checkpoints = sorted(
        [d for d in os.listdir(checkpoint_root) if d.startswith("checkpoint-") and os.path.isdir(os.path.join(checkpoint_root, d))],
        key=lambda x: int(x.split("-")[-1]),
        reverse=True
    )

    for checkpoint_name in checkpoints:
        trainer_state_path = os.path.join(checkpoint_root, checkpoint_name, "trainer_state.json")
        if os.path.isfile(trainer_state_path):
            with open(trainer_state_path, "r") as f:
                trainer_state = json.load(f)
            log_history = trainer_state.get("log_history", [])
            if log_history:
                os.makedirs(output_dir, exist_ok=True)
                save_path = os.path.join(output_dir, "training_metrics.png")
                plot_all_metrics(log_history, save_path)
                return
            else:
                print(f"⚠️  trainer_state.json in {checkpoint_name} has no log_history.")
        else:
            print(f"❌ No trainer_state.json in: {checkpoint_name}")
    print("❌ No valid trainer_state.json found with logs in any checkpoint-* folder.")

def plot_combined_training_validation_reward(training_json_path, validation_json_path, output_dir):
    """
    Plots both training and validation average rewards per epoch in a single chart.
    
    Args:
        training_json_path (str): Path to training_log.json
        validation_json_path (str): Path to validation_log.json  
        output_dir (str): Directory to save the plot
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Load training data
    with open(training_json_path, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    
    # Handle both list and dict formats for training data
    if isinstance(train_data, list):
        epochs_data = {}
        for record in train_data:
            epoch = record.get("epoch", 0)
            if epoch not in epochs_data:
                epochs_data[epoch] = []
            paper = {
                "title of paper": f"paper_{record.get('record_id', 'unknown')}",
                "generated_ideas": [{
                    "generated_idea": record.get("completion", ""),
                    "reward": record.get("reward", 0)
                }]
            }
            epochs_data[epoch].append(paper)
        train_data = {"epochs": epochs_data}
    
    # Calculate training rewards
    train_avg_rewards = []
    train_epochs = []
    for epoch_str, papers in train_data.get("epochs", {}).items():
        rewards = [idea["reward"] for paper in papers for idea in paper["generated_ideas"]]
        if rewards:
            train_avg_rewards.append(sum(rewards) / len(rewards))
            train_epochs.append(int(epoch_str))
    
    # Load validation data if it exists
    val_avg_rewards = []
    val_epochs = []
    if os.path.exists(validation_json_path):
        with open(validation_json_path, 'r', encoding='utf-8') as f:
            val_data = json.load(f)
        
        # Validation data structure: {epoch: [records]} where each record has 'reward' directly
        for run_str, records in val_data.items():
            rewards = [record["reward"] for record in records if "reward" in record]
            if rewards:
                val_avg_rewards.append(sum(rewards) / len(rewards))
                val_epochs.append(float(run_str))
    
    # Create the combined plot
    plt.figure(figsize=(12, 6))
    
    if train_epochs and train_avg_rewards:
        plt.plot(train_epochs, train_avg_rewards, marker='o', linestyle='-', 
                color='blue', label='Training Reward', linewidth=2)
    
    if val_epochs and val_avg_rewards:
        plt.plot(val_epochs, val_avg_rewards, marker='s', linestyle='--', 
                color='red', label='Validation Reward', linewidth=2)
    
    # Set y-axis limits
    all_rewards = train_avg_rewards + val_avg_rewards
    if all_rewards:
        plt.ylim(0, max(all_rewards) * 1.1)
    
    plt.xlabel("Epoch")
    plt.ylabel("Average Reward")
    plt.suptitle(training_json_path.split("/")[1])
    plt.title("Training vs Validation Average Reward per Epoch")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save the plot
    combined_path = os.path.join(output_dir, "combined_train_val_reward.png")
    plt.savefig(combined_path)
    print(f"✅ Saved combined training/validation reward plot to {combined_path}")
    plt.close()

# Plot

In [8]:
# 🎯 Main wrapper to generate all plots
def generate_all_run_plots(run_dir):
    training_log_path = os.path.join(run_dir, "training_log.json")
    validation_log_path = os.path.join(run_dir, "validation_log.json")
    plots_dir = os.path.join(run_dir, "plots")

    if not os.path.exists(training_log_path):
        print(f"❌ training_log.json not found in: {run_dir}")
        return

    print(f"\n📊 Generating plots for run: {run_dir}")
    plot_all_training_stats(training_log_path, plots_dir)
    if os.path.exists(validation_log_path):
        plot_all_validation_stats(validation_log_path, plots_dir)
    else:
        print(f"⚠️ validation_log.json not found in: {run_dir}")
    plot_combined_training_validation_reward(training_log_path, validation_log_path, plots_dir)
    plot_avg_and_moving_reward_per_step(training_log_path, plots_dir)
    plot_from_latest_checkpoint(run_dir, plots_dir)
    plot_reward_transitions_combined(training_log_path, plots_dir)
    plot_reward_distribution_per_step(training_log_path, plots_dir)
    plot_reward_variance_line(training_log_path, plots_dir)
    print(f"✅ All plots saved in: {plots_dir}")

In [9]:
import os
def generate_all_runs_in_results(root_dir="results"):
    """
    Iterates over all run directories inside `root_dir` and generates all training plots.
    Assumes each subfolder is a training run with `training_log.json` and `checkpoint/`.

    Args:
        root_dir (str): Path to the root results directory containing run subfolders.
    """
    if not os.path.isdir(root_dir):
        print(f"❌ '{root_dir}' is not a valid directory.")
        return

    run_dirs = [os.path.join(root_dir, name) for name in os.listdir(root_dir)
                if os.path.isdir(os.path.join(root_dir, name))]

    print(f"🔍 Found {len(run_dirs)} run directories in '{root_dir}'")
    num =0
    for run_dir in run_dirs:
        num +=1
        if num == 1:
            continue
        training_log_path = os.path.join(run_dir, "training_log.json")
        if os.path.isfile(training_log_path):
            if os.path.getmtime(training_log_path) < VISUALIZE_MODIF_TIME and False:
                print(f"Skipping '{run_dir}': no recent changes were found.")
                continue
            try:
                generate_all_run_plots(run_dir)
            except Exception as e:
                print(f"❌ Failed to generate‍ plots for '{run_dir}': {e}")
        else:
            print(f"⚠️ Skipping '{run_dir}': training_log.json not found.")


In [10]:
generate_all_runs_in_results()

🔍 Found 2 run directories in 'results'

📊 Generating plots for run: results/Training_dt11.18.17:40_e20_unsloth_Qwen2.5_3B_Instruct_unsloth_bnb_4bit_bnb_4bit_lr1e-05_t0.7_ε0.2_r64_b16
❌ Could not extract batch size from path.
⚠️ Could not extract batch size from path, using default value of 8.
✅ Saved validation reward plot to results/Training_dt11.18.17:40_e20_unsloth_Qwen2.5_3B_Instruct_unsloth_bnb_4bit_bnb_4bit_lr1e-05_t0.7_ε0.2_r64_b16/plots/avg_validation_reward.png
✅ Saved combined training/validation reward plot to results/Training_dt11.18.17:40_e20_unsloth_Qwen2.5_3B_Instruct_unsloth_bnb_4bit_bnb_4bit_lr1e-05_t0.7_ε0.2_r64_b16/plots/combined_train_val_reward.png
⚠️ Could not extract batch size from path, using default value of 8.
✅ Saved reward-per-step plot to results/Training_dt11.18.17:40_e20_unsloth_Qwen2.5_3B_Instruct_unsloth_bnb_4bit_bnb_4bit_lr1e-05_t0.7_ε0.2_r64_b16/plots/reward_per_step.png
📈 Plot saved to: results/Training_dt11.18.17:40_e20_unsloth_Qwen2.5_3B_Instruct_